<a href="https://colab.research.google.com/github/hgmhd7/AI-Machine-Learning/blob/main/ShopTalk_ItemCentric_BLIP_CLIP_Pipeline_v9_7_blip_text_cleanup_abtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ShopTalk — Item-Centric Multimodal Pipeline (v6)
This notebook generates **item_id-centric** retrieval artifacts from your product images + metadata.

**What you get at the end (per EMBEDDING_MODEL_VERSION):**
- `master_metadata_enriched` (**Parquet + CSV**) — 1 row per `embedding_id`
- `clip_image_embeddings_table` (**Parquet + CSV**) — image-centric table (1 row per `embedding_id`)
- `item_level_index` (**Parquet + CSV**) — 1 row per `item_id` (aggregated embeddings)
- FAISS indices:
  - `faiss_item.index` (+ `faiss_item_idmap.csv`)
  - `faiss_image.index` (+ `faiss_image_idmap.csv`, image-centric using `image_name`)

**Design principles**
- `embedding_id` is unique per row (per image instance).
- `image_name` is the primary image identity for image-centric search workflows.
- `item_id` is the primary product identity for shopping search workflows.
- Outputs are **versioned** by `EMBEDDING_MODEL_VERSION` so you can safely regenerate with new models/settings later.

---

## Step-by-step outline (numbered)
1. Setup + versioned output folders (Parquet default, also export CSV)
2. Load master metadata + sanity checks
3. Identify hero image per item (`is_hero`)
4. BLIP: conditional captions (metadata-conditioned)
5. CLIP: caption embeddings (BLIP caption → CLIP text embedding)
6. CLIP: image embeddings (image → CLIP image embedding) + image-centric table
7. Item aggregation: hero-weighted caption embedding centroid per item
8. Build FAISS indices (item-level + image-level)
9. Quick retrieval demos (text query → items, optional image query later)


## How to run this notebook (recommended)
1. Run **Step 1–3** to confirm paths and schema.
2. Run **Step 4–7** to generate BLIP captions (long-running; safe to resume).
3. Run **Step 8–11** to generate CLIP embeddings + item aggregation (long-running; safe to resume).
4. Run **Step 12–14** to build FAISS indices and sanity-check retrieval.

## Interruption/resume guidance
- If Colab disconnects:
  - re-run Step 1–3
  - then rerun the long steps (captions/embeddings). They will skip `embedding_id`s already processed because files/rows already exist.


### Runtime note (AWS-local Colab)
This notebook is configured for a local Colab instance connected to AWS.
Root data directory:
`/home/jovyan/data/shop_talk_data`


## End-to-end data lineage example (one image → one item)

This is the “mental model” for how a single row flows through the pipeline.

### Starting point (from `master_metadata`)
One **row** represents one image instance:

- `embedding_id = "E123456789"`  *(unique per row)*
- `item_id = "ITEM_98765"`        *(product identifier)*
- `image_name = "B00ABCDEF0_001"` *(image identity, no extension)*
- `image_path = ".../images/B00ABCDEF0_001.jpg"`
- `blip_text_input = "red short sleeve shirt, men's, under $50, cotton"` *(example prompt)*

### Step-by-step outputs produced for this row
1. **BLIP conditional caption**
   - `blip_caption = "red short-sleeve cotton shirt"`

2. **CLIP caption embedding** *(searchable in CLIP text space)*
   - saved to: `clip_caption_embeddings/E123456789.npy`
   - linked from table via: `clip_caption_embedding_path`

3. **CLIP image embedding** *(searchable in CLIP image space)*
   - saved to: `clip_image_embeddings/E123456789.npy`
   - linked from image table via: `clip_image_embedding_path`

### How it contributes to the item (`item_id`)
For `ITEM_98765`, you’ll have multiple image rows (multiple `embedding_id`s). We compute:

- `item_embedding = hero_weighted_mean( caption_embeddings_for_all_images_in_item )`
- saved to: `item_embeddings/ITEM_98765.npy`

### What your application searches
- **Text query → item retrieval**: CLIP(text query) → `faiss_item.index` → top `item_id`s
- **Image query → image retrieval**: CLIP(image) → `faiss_image.index` → top `image_name`s → map to `item_id`s

This design gives you:
- product-level retrieval (what users want to buy)
- image-level similarity (better UX and reranking)
- clean traceability (every vector ties back to `embedding_id`)


## Docker / API deployment alignment (recommended)

If you plan to serve this via an API later (FastAPI/Flask), keep **the same artifacts** produced here and only add a thin service layer.

### Suggested on-disk layout (works both locally and in Docker)
- `shop_talk_data/`
  - `product_data/`
    - `master_metadata.csv`
    - `images/` *(or your existing image folders)*
  - `model_outputs/`
    - `<EMBEDDING_MODEL_VERSION>/`
      - tables: `*.parquet` + `*.csv`
      - embeddings: `clip_*_embeddings/*.npy`, `item_embeddings/*.npy`
      - faiss: `faiss/*.index` + `*_idmap.csv`
  - `services/` *(API code; optional)*
    - `app/` (FastAPI)
    - `config/`

### Docker tips
- Use environment variables so the same code runs everywhere:
  - `SHOP_TALK_ROOT=/home/jovyan/data/shop_talk_data`
  - `EMBEDDING_MODEL_VERSION=...`
- Mount your data directory into the container:
  - `-v /home/jovyan/data/shop_talk_data:/data/shop_talk_data`
- Then set `BASE_DIR='/data/shop_talk_data'` inside Docker.

### What you **won't** need to rebuild later
- Captions, embeddings, Parquet/CSV tables, FAISS indices — all of these are loadable directly by the API.


## Additions (v4)
This notebook now also includes:

1) **Versioned embeddings** via `EMBEDDING_MODEL_VERSION`
- Embeddings and indices are written under versioned subfolders so you can re-run with new models/settings safely.

2) **FAISS indices**
- `faiss_item.index` for fast item-level retrieval
- Optional `faiss_image.index` for image-level similarity / reranking
- ID mapping files saved alongside each index so you can convert FAISS result positions back to `item_id` / `embedding_id`.


In [ ]:
# ====== 0) Colab/Drive Setup (edit paths as needed) ======
# (AWS-local) Google Drive mount not used
import os, pandas as pd

BASE_DIR = '/home/jovyan/data/shop_talk_data'
MASTER_METADATA_PATH = os.path.join(BASE_DIR, 'product_data', 'master_metadata.csv')

OUT_DIR = os.path.join(BASE_DIR, 'model_outputs_v3_itemcentric')
os.makedirs(OUT_DIR, exist_ok=True)

# Table outputs
MASTER_ENRICHED_OUT = os.path.join(OUT_DIR, 'master_metadata_enriched.csv')
CLIP_IMAGE_TABLE_OUT = os.path.join(OUT_DIR, 'clip_image_embeddings_table.csv')
ITEM_LEVEL_OUT = os.path.join(OUT_DIR, 'item_level_index.csv')

# Embedding directories
CLIP_CAPTION_EMB_DIR = os.path.join(OUT_DIR, 'clip_caption_embeddings')   # per embedding_id .npy
CLIP_IMAGE_EMB_DIR   = os.path.join(OUT_DIR, 'clip_image_embeddings')     # per embedding_id .npy
ITEM_EMB_DIR         = os.path.join(OUT_DIR, 'item_embeddings')           # per item_id .npy

for d in [CLIP_CAPTION_EMB_DIR, CLIP_IMAGE_EMB_DIR, ITEM_EMB_DIR]:
    os.makedirs(d, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('OUT_DIR:', OUT_DIR)
print('MASTER_METADATA_PATH:', MASTER_METADATA_PATH)


# ====== Versioning ======
# Change this when you change models, prompting, token limits, or any embedding recipe.
EMBEDDING_MODEL_VERSION = 'v1_clip-vit-b32__blip-base__caption2clip'

# Versioned subfolders
VERSION_DIR = os.path.join(OUT_DIR, EMBEDDING_MODEL_VERSION)
os.makedirs(VERSION_DIR, exist_ok=True)

# Override outputs into versioned folder (keeps runs clean)
MASTER_ENRICHED_OUT = os.path.join(VERSION_DIR, 'master_metadata_enriched.csv')
CLIP_IMAGE_TABLE_OUT = os.path.join(VERSION_DIR, 'clip_image_embeddings_table.csv')
ITEM_LEVEL_OUT = os.path.join(VERSION_DIR, 'item_level_index.csv')

CLIP_CAPTION_EMB_DIR = os.path.join(VERSION_DIR, 'clip_caption_embeddings')
CLIP_IMAGE_EMB_DIR   = os.path.join(VERSION_DIR, 'clip_image_embeddings')
ITEM_EMB_DIR         = os.path.join(VERSION_DIR, 'item_embeddings')

for d in [CLIP_CAPTION_EMB_DIR, CLIP_IMAGE_EMB_DIR, ITEM_EMB_DIR]:
    os.makedirs(d, exist_ok=True)

# FAISS outputs
FAISS_DIR = os.path.join(VERSION_DIR, 'faiss')
os.makedirs(FAISS_DIR, exist_ok=True)

FAISS_ITEM_INDEX_PATH  = os.path.join(FAISS_DIR, 'faiss_item.index')
FAISS_ITEM_IDMAP_PATH  = os.path.join(FAISS_DIR, 'faiss_item_idmap.csv')

FAISS_IMAGE_INDEX_PATH = os.path.join(FAISS_DIR, 'faiss_image.index')
FAISS_IMAGE_IDMAP_PATH = os.path.join(FAISS_DIR, 'faiss_image_idmap.csv')

print('EMBEDDING_MODEL_VERSION:', EMBEDDING_MODEL_VERSION)
print('VERSION_DIR:', VERSION_DIR)
print('FAISS_DIR:', FAISS_DIR)


# ====== Table output formats ======
# We write Parquet by default (fast, typed, compressed) AND also export CSV for human readability.
MASTER_ENRICHED_OUT_CSV   = MASTER_ENRICHED_OUT
MASTER_ENRICHED_OUT_PQ    = MASTER_ENRICHED_OUT.replace('.csv', '.parquet')

CLIP_IMAGE_TABLE_OUT_CSV  = CLIP_IMAGE_TABLE_OUT
CLIP_IMAGE_TABLE_OUT_PQ   = CLIP_IMAGE_TABLE_OUT.replace('.csv', '.parquet')

ITEM_LEVEL_OUT_CSV        = ITEM_LEVEL_OUT
ITEM_LEVEL_OUT_PQ         = ITEM_LEVEL_OUT.replace('.csv', '.parquet')

print('Parquet outputs will be written to:')
print(' -', MASTER_ENRICHED_OUT_PQ)
print(' -', CLIP_IMAGE_TABLE_OUT_PQ)
print(' -', ITEM_LEVEL_OUT_PQ)


BASE_DIR: /home/jovyan/data/shop_talk_data
OUT_DIR: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric
MASTER_METADATA_PATH: /home/jovyan/data/shop_talk_data/product_data/master_metadata.csv
EMBEDDING_MODEL_VERSION: v1_clip-vit-b32__blip-base__caption2clip
VERSION_DIR: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip
FAISS_DIR: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/faiss
Parquet outputs will be written to:
 - /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/master_metadata_enriched.parquet
 - /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/clip_image_embeddings_table.parquet
 - /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/item_level_index.parquet


In [ ]:
# ====== Step 1a) Dependencies (FAISS + Parquet engines) ======
# Why this cell exists:
# - Colab runtimes can start "empty" and not have parquet engines installed.
# - We install what we need up front so later steps don't fail midway.
#
# Notes:
# - `pyarrow` is the most common Parquet engine.
# - `faiss-cpu` is installed later too, but we keep it here for consistency.
# - If you're on a GPU runtime, `faiss-gpu` exists, but `faiss-cpu` is fine for indexing moderate-sized datasets.

!pip -q install pyarrow fastparquet faiss-cpu
print("Installed: pyarrow, fastparquet, faiss-cpu")


Installed: pyarrow, fastparquet, faiss-cpu


In [ ]:
# ====== 1) Load master metadata + sanity checks ======
import pandas as pd

master_metadata_df = pd.read_csv(MASTER_METADATA_PATH)

required_cols = ['item_id','embedding_id','image_id','image_name','image_path','blip_text_input']
missing = [c for c in required_cols if c not in master_metadata_df.columns]
assert not missing, f"Missing required columns: {missing}"

# Normalize types
master_metadata_df['embedding_id'] = master_metadata_df['embedding_id'].astype(str)

print('Rows:', len(master_metadata_df))
print('Unique item_id:', master_metadata_df['item_id'].nunique())
print('Unique embedding_id:', master_metadata_df['embedding_id'].nunique())
print(master_metadata_df[required_cols].head(3))


Rows: 133833
Unique item_id: 132360
Unique embedding_id: 133833
      item_id     embedding_id     image_id image_name  \
0  B07BMPCX9R  A1k2-zO8c+L__JP  A1k2-zO8c+L   72e64ca0   
1  B06X6J9TFJ  71WrrK6eaNL__IN  71WrrK6eaNL   0da379d4   
2  B075GWYJRC  71ZVZEAW3sL__GB  71ZVZEAW3sL   eaa361cd   

                                          image_path  \
0  /home/jovyan/data/shop_talk_data/image_data/im...   
1  /home/jovyan/data/shop_talk_data/image_data/im...   
2  /home/jovyan/data/shop_talk_data/image_data/im...   

                                     blip_text_input  
0  /カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime...  
1  /Categories/Kitchen & Dining/Kitchen Storage &...  
2  Cargo Express furniture; express furniture; ex...  


## Step 3A — Clean `blip_text_input` (baseline prompt cleanup)

**Goal**: Improve prompt quality before the A/B caption test and full caption generation.

We keep the original column intact and create a new column:

- `blip_text_input_clean` → cleaned version used for BLIP conditioning

**What we clean**
- Remove slashes (`/`)
- Remove the word “caption” and common variants (caption, captions, captioning, captioned)
- Remove label-y boilerplate prefixes (Title:, Product:, Description:, Bullet Points:)
- Normalize separators (`|` and `;` → `,`)
- Normalize whitespace
- Truncate to a safe length (default 300 chars) to avoid “attribute soup” prompts

We also print basic stats so you can see how aggressive truncation is.


In [ ]:
import re
import pandas as pd

# ---- Configuration ----
MAX_PROMPT_CHARS = 2000  # keep prompts concise; adjust after reviewing stats

def clean_blip_text(s: str) -> str:
    # Baseline cleaning for BLIP conditioning text.
    if s is None:
        return ""
    s = str(s)

    # 1) Remove slashes
    s = s.replace("/", " ")

    # 2) Remove 'caption' and common variations (case-insensitive), plus trailing punctuation
    # Matches: caption, captions, captioning, captioned (optionally followed by : - – —)
    s = re.sub(r"\bcaption(?:s|ing|ed)?\b\s*[:\-–—]*\s*", "", s, flags=re.IGNORECASE)

    # 3) Remove common label-y prefixes that make prompts non-conversational
    s = re.sub(r"\b(title|product|description|bullet points?)\b\s*[:\-–—]\s*", "", s, flags=re.IGNORECASE)

    # 4) Normalize separators
    s = s.replace("|", ",").replace(";", ",")

    # 5) Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # 6) Truncate (helps BLIP stay natural)
    if len(s) > MAX_PROMPT_CHARS:
        s = s[:MAX_PROMPT_CHARS].rstrip()

    return s

# Create additive cleaned column (do not overwrite raw)
master_metadata_df["blip_text_input_clean"] = master_metadata_df["blip_text_input"].apply(clean_blip_text)

# ---- Quick stats ----
raw_len = master_metadata_df["blip_text_input"].fillna("").astype(str).str.len()
clean_len = master_metadata_df["blip_text_input_clean"].fillna("").astype(str).str.len()

print("BLIP text length stats (raw)  :", raw_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_dict())
print("BLIP text length stats (clean):", clean_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_dict())

# Show a few before/after examples
display_cols = ["embedding_id", "item_id", "image_name", "blip_text_input", "blip_text_input_clean"]
print("\nSample before/after (first 5 rows):")
display(master_metadata_df[display_cols].head(5))


BLIP text length stats (raw)  : {'count': 133833.0, 'mean': 324.5086040064857, 'std': 104.37563750184236, 'min': 22.0, '50%': 338.0, '90%': 426.0, '95%': 465.0, '99%': 609.0, 'max': 1501.0}
BLIP text length stats (clean): {'count': 133833.0, 'mean': 323.3713882226357, 'std': 104.19704254609412, 'min': 22.0, '50%': 337.0, '90%': 425.0, '95%': 463.0, '99%': 608.0, 'max': 1496.0}

Sample before/after (first 5 rows):


,embedding_id,item_id,image_name,blip_text_input,blip_text_input_clean
0,A1k2-zO8c+L__JP,B07BMPCX9R,72e64ca0,"/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate","カテゴリー別 缶詰・瓶詰 肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate"
1,71WrrK6eaNL__IN,B06X6J9TFJ,0da379d4,"/Categories/Kitchen & Dining/Kitchen Storage & Containers/Water Bottles bottles 1litre,cool water bottle,fridge bottle,fridge bottles,fridge water bottles,hot water bottles,stainless steel bottle,stainless steel bottles,stainless steel water bottle,stainless steel water bottles,stainless steel water bottles 1000ml,stainless steel water bottles 500ml,steel bottle,steel bottle for kids,steel bottles,steel bottles for water 1 liter,steel fridge water bottle,steel water bottle,steel water bottle 1 litre,steel water bottle 1000ml,steel water bottle 750 ml,steel water bottle 750ml,steel water bottle for kids,steel water bottle kids,steel water bottles,stylish water bottle,watar bottle,watee bottle,water bottle,water bottle 1 litre,water bottle 1liter,water bottle 1liter steel,water bottle for office,water bottle h2o,water bottle stainless stee

## Step 1 — Environment + paths (versioned outputs)

**What this step does**
- Mounts Google Drive (Colab) so we can read the master table and write outputs that survive runtime resets.
- Defines **one root output folder** and then a **versioned subfolder** (`EMBEDDING_MODEL_VERSION`).
- Creates a consistent on-disk layout for:
  - CSV tables (human readable, resumable appends)
  - Parquet tables (canonical typed snapshots)
  - `.npy` embeddings (fast binary vectors)
  - FAISS index files + ID maps

**Why it matters**
- If Colab disconnects, you can re-run the notebook and it will **pick up where it left off** because outputs are written to Drive.
- Versioning prevents you from accidentally mixing vectors from different model runs.

**You typically edit only these values**
- `BASE_DIR`
- `MASTER_METADATA_PATH`
- `EMBEDDING_MODEL_VERSION`


### Inputs
- Drive location (`BASE_DIR`)
- Master table path (`MASTER_METADATA_PATH`)
- `EMBEDDING_MODEL_VERSION`

### Outputs
- Versioned folder layout under `VERSION_DIR`
- Output path variables for:
  - CSV tables (checkpoint)
  - Parquet tables (canonical)
  - `.npy` embeddings
  - FAISS indices + id-maps

In [ ]:
# ====== Step 1) 1b) Stamp embedding version into dataframe (for traceability) ======
master_metadata_df['embedding_model_version'] = EMBEDDING_MODEL_VERSION
print(master_metadata_df[['embedding_id','item_id','image_name','embedding_model_version']].head(3))


      embedding_id     item_id image_name  \
0  A1k2-zO8c+L__JP  B07BMPCX9R   72e64ca0   
1  71WrrK6eaNL__IN  B06X6J9TFJ   0da379d4   
2  71ZVZEAW3sL__GB  B075GWYJRC   eaa361cd   

                    embedding_model_version  
0  v1_clip-vit-b32__blip-base__caption2clip  
1  v1_clip-vit-b32__blip-base__caption2clip  
2  v1_clip-vit-b32__blip-base__caption2clip  


## Step 2 — Dependencies (Parquet + FAISS)

**What this step does**
- Installs the packages required for:
  - Writing Parquet (`pyarrow`, `fastparquet`)
  - Building/searching FAISS indices (`faiss-cpu`)

**Why it matters**
- Colab images vary; installing up front prevents “halfway through the run” failures.
- This is safe to run multiple times (pip will just confirm packages are present).


### Inputs
- None (fresh Colab runtime)

### Outputs
- Installed Python packages:
  - Parquet engines: `pyarrow`, `fastparquet`
  - ANN search: `faiss-cpu`

In [ ]:
# ====== Step 2) Helper: export Parquet snapshot from a CSV table ======
# Commentary:
# - CSV is our interruption-safe checkpoint format (append-friendly).
# - Parquet is our canonical analytics/storage format (typed, compressed, fast).
# - This helper converts the latest CSV checkpoint into a Parquet snapshot whenever we finish a stage.

import pandas as pd

def export_parquet_from_csv(csv_path: str, parquet_path: str):
    """Read CSV (human readable) and write Parquet (fast + typed) snapshot."""
    df = pd.read_csv(csv_path)
    df.to_parquet(parquet_path, index=False)
    print('Wrote Parquet:', parquet_path, '| rows:', len(df))

In [ ]:
# ====== 2) Identify hero image per item (is_hero) ======
# Primary: main_image_id matches image_id
# Fallback: first row encountered per item_id
import numpy as np

master_metadata_df['is_hero'] = False

if 'main_image_id' in master_metadata_df.columns:
    # Mark rows where image_id == main_image_id
    master_metadata_df.loc[master_metadata_df['image_id'] == master_metadata_df['main_image_id'], 'is_hero'] = True

# If no hero marked for an item, mark first row as hero
hero_counts = master_metadata_df.groupby('item_id')['is_hero'].sum()
items_missing_hero = hero_counts[hero_counts == 0].index.tolist()

if items_missing_hero:
    idx = (master_metadata_df[master_metadata_df['item_id'].isin(items_missing_hero)]
           .groupby('item_id', sort=False)
           .head(1)
           .index)
    master_metadata_df.loc[idx, 'is_hero'] = True

print('Items missing hero after fix:', int((master_metadata_df.groupby('item_id')['is_hero'].sum()==0).sum()))
print(master_metadata_df[['item_id','image_id','main_image_id','image_name','is_hero']].head(10))


Items missing hero after fix: 0
      item_id     image_id main_image_id image_name  is_hero
0  B07BMPCX9R  A1k2-zO8c+L   A1k2-zO8c+L   72e64ca0     True
1  B06X6J9TFJ  71WrrK6eaNL   71WrrK6eaNL   0da379d4     True
2  B075GWYJRC  71ZVZEAW3sL   71ZVZEAW3sL   eaa361cd     True
3  B07Q56TV7J  61imkosX-IL   61imkosX-IL   a2ac43e7     True
4  B081Z3GZT6  61r651X8LbL   61r651X8LbL   e3d18f31     True
5  B07TP692HX  61Gs3w501SL   61Gs3w501SL   fae3406c     True
6  B07YJV8CC9  71XaT-wEe+L   71XaT-wEe+L   c7313a1a     True
7  B089LPH5B2  71pR-qSke2L   71pR-qSke2L   f10bc7ef     True
8  B06XXQXD5W  91gM-ctvLzL   91gM-ctvLzL   4721ef45     True
9  B07G58NNWS  51HEbUluSLL   51HEbUluSLL   6e7afdae     True


In [ ]:
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade transformers accelerate safetensors


In [ ]:
!pip -q install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade transformers accelerate safetensors


In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.5.1+cu121
CUDA available: True


In [ ]:
# ====== 3) BLIP (conditional captioning) setup ======
# Commentary:
# - BLIP here is used in *conditional captioning* mode.
# - We feed BLIP two inputs:
#   (1) the product image
#   (2) `blip_text_input` (metadata prompt) to steer caption generation toward relevant attributes.
# - The output is a short caption string we later embed in CLIP text space for retrieval.
#
# NOTE (IMPORTANT):
# - We explicitly disable the "fast" image processor (use_fast=False)
# - We force safetensors loading (use_safetensors=True)
#   This avoids recent torch>=2.6 restrictions while remaining functionally identical.

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)
print('torch:', torch.__version__)

# Force the slow processor to avoid fast-backend requirements
blip_processor = BlipProcessor.from_pretrained(
    'Salesforce/blip-image-captioning-base',
    use_fast=False
)

# Force safetensors loading to avoid torch.load(weights_only=True)
blip_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base',
    use_safetensors=True
).to(DEVICE)

blip_model.eval()
print("✅ BLIP loaded successfully.")

def generate_image_caption(image_path: str, max_new_tokens: int = 30) -> str:
    """Caption using image only (no metadata prompt)."""
    image = Image.open(image_path).convert('RGB')
    inputs = blip_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=max_new_tokens)
    caption = blip_processor.decode(out[0], skip_special_tokens=True).strip()
    return caption


def generate_conditional_caption(
    image_path: str,
    prompt_text: str,
    max_new_tokens: int = 30
) -> str:
    """Caption conditioned on metadata prompt_text + image."""
    image = Image.open(image_path).convert('RGB')
    inputs = blip_processor(
        images=image,
        text=prompt_text,
        return_tensors='pt'
    ).to(DEVICE)

    with torch.no_grad():
        out = blip_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    caption = blip_processor.decode(
        out[0],
        skip_special_tokens=True
    ).strip()

    return caption

DEVICE: cuda
torch: 2.5.1+cu121


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BLIP loaded successfully.


## Step 3A — A/B caption test (image-only vs metadata-conditioned)

Before generating captions for the whole dataset, this quick test generates two captions per image for a small sample:

- **A (image-only)**: BLIP sees only the image.
- **B (conditioned)**: BLIP sees the image **plus** `blip_text_input` as a steering prompt.

**Why**: This lets you quickly judge whether conditioning makes captions more natural/helpful or more “stiff”.


In [ ]:
# Create additive cleaned column (do not overwrite raw)
master_metadata_df["blip_text_input_clean"] = master_metadata_df["blip_text_input"].apply(clean_blip_text)

In [ ]:
# ====== 3A) A/B caption test (image-only vs conditioned) ======

import pandas as pd
import numpy as np
import os

AB_SAMPLE_N = 100
AB_RANDOM_SEED = 42
AB_OUT_CSV = os.path.join(VERSION_DIR, "ab_caption_test.csv")

# Use the full master metadata (includes country, locale, etc.)
ab_df = master_metadata_df.copy()
ab_df["embedding_id"] = ab_df["embedding_id"].astype(str)

# Sample rows that actually have image paths present
ab_df = ab_df[ab_df["image_path"].notna()].copy()

sample_df = ab_df.sample(n=min(AB_SAMPLE_N, len(ab_df)), random_state=AB_RANDOM_SEED).reset_index(drop=True)

rows = []
for row in sample_df.itertuples(index=False):
    try:
        cap_img = generate_image_caption(row.image_path)
        err_img = ""
    except Exception as e:
        cap_img = ""
        err_img = str(e)

    try:
        cap_cond = generate_conditional_caption(row.image_path, str(row.blip_text_input_clean))
        err_cond = ""
    except Exception as e:
        cap_cond = ""
        err_cond = str(e)

    rows.append({
        "embedding_id": str(row.embedding_id),
        "item_id": getattr(row, "item_id", None),
        "image_name": getattr(row, "image_name", None),
        "country": getattr(row, "country", None),
        "blip_text_input": str(getattr(row, "blip_text_input", "")),
        "caption_A_image_only": cap_img,
        "caption_B_conditioned": cap_cond,
        "error_A": err_img,
        "error_B": err_cond,
    })

ab_out = pd.DataFrame(rows)
ab_out.to_csv(AB_OUT_CSV, index=False)
print("Wrote A/B caption test:", AB_OUT_CSV)
ab_out.head(10)

Wrote A/B caption test: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/ab_caption_test.csv


,embedding_id,item_id,image_name,country,blip_text_input,caption_A_image_only,caption_B_conditioned,error_A,error_B
0,71H15JG9t-L__IN,B08545ZY6B,5113fcba,IN,"Vivo Z1 Pro /Categories/Mobiles & Accessories/Mobile Accessories/Maintenance, Upkeep & Repairs/Replacement Parts/Back Covers Back Cover Amazon Brand - Solimo Amazon Brand - Solimo Designer Photography UV Printed Soft Back Case Mobile Cover for Vivo Z1 Pro CELLULAR_PHONE_CASE Multicolor Silicon Snug fit for Vivo Z1 Pro, with perfect cut-outs for volume buttons, audio and charging ports",a phone case with a unicorn and a unicorn on it,"vivo z1 pro categories mobiles & accessories mobile accessories maintenance, upkeep & repairs replacement parts back covers back cover amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro cellular _ phone _ case multicolor silicon snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports",,
1,915+S1MRS4L__SG,B07PHFWND8,fa72077d,SG,"Sheet Set - Twin, Sage Vine /Homeware & Furniture/Bedding/Sheets & Pillowcases/Sheet & Pillowcase Sets bed AmazonBasics AmazonBasics Super-Soft Cotton Bed Sheet Set-Twin, Sage Vine FLAT_SHEET Sage Vine 100% Cotton Twin set contains one 68 x 96 inch flat sheet; one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",the white and green floral bedding set,"sheet set - twin, sage vine homeware & furniture bedding sheets & pillowcases sheet & pillowcase sets bed amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine flat _ sheet sage vine 100 % cotton twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",,
2,617Gg5OD-TL__IN,B0856G38P5,54462c3c,IN,"Lenovo P2 /Categories/Mobiles & Accessories/Mobile Accessories/Cases & Covers/Back & Bumper Cases cellphonecover Amazon Brand - Solimo Amazon Brand - Solimo Designer Heart Pattern Alphabet-H 3D Printed Hard Back Case Mobile Cover for Lenovo P2 CELLULAR_PHONE_CASE multi-colored Plastic Snug fit for Mobile, with perfect cut-outs for volume buttons, audio and charging ports",the h logo on a pink and black phone case,"lenovo p2 categories mobiles & accessories mobile accessories cases & covers back & bumper cases cellphonecover amazon brand - solimo amazon brand - solimo designer heart pattern alphabet - h 3d printed hard back case mobile cover for lenovo p2 cellular _ phone _ case multi - colored plastic snug fit for mobile, with perfect cut - outs for volume buttons, audio and charging ports",,
3,71u7QAUnzkL__IN,B07TRWZ3N2,0b61a49e,IN,Xiaomi Redmi Note 7 Pro /Categories/Mobiles & Accessories/Mobile Accessories/Cases & Covers/Back & Bumper Cases mobile cover Amazon Brand - Solimo Amazon Brand - Solimo Designer I Love U 3D Printed Hard Back Case Mobile Cover for Xiaomi Redmi Note 7 Pro CELLULAR_PHONE_CASE Others 3D Printed Hard Back Case Mobile Cover for Xiaomi Redmi Note 7 Pro,a pink phone case with a bird on it,xiaomi redmi note 7 pro categories mobiles & accessories mobile accessories cases & covers back & bumper cases mobile cover amazon brand - solimo amazon brand - solimo designer i love u 3d printed hard back case mobile cover for xiaomi redmi note 7 pro cellular _ phone _ case others 3d printed hard back case mobile cover for xiaomi redmi note 7 pro,,
4,71kliiJG1BL__IN,B07TC5XF1B,b314eda4,IN,Meizu M3 Note /Categories/Mobiles & Accessories/Mobile Accessories/Cases & Covers/Back & Bumper Cases mobile cover Amazon Brand - Solimo Amazon Brand - Solimo Designer Summer Juice 3D Printed Hard Back Case Mobile Cover for Meizu M3 Note CELLULAR_PHONE_CASE Others 3D Printed Hard Back Case Mobile Cover for Meizu M3 Note,the summer cocktail phone case,meizu m3 note categories mobiles & accessories mobile accessories cases & covers back & bumper cases mobile cover amazon brand - solimo amazon brand - solimo designer summer juice 3d printed hard back case mobile cover for meizu m3 note c

## Test Image vs Conditioned Text

In [ ]:
# Filter the ab_out df by the rows that have US as the country
ab_out_us = ab_out[ab_out['country'] == 'US']

# Make sure the entire datrafram is visable in the results window
# I need to be able to see the entire dataframe in the results window
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Check df and show all rows
ab_out_us.head(100)

,embedding_id,item_id,image_name,country,blip_text_input,caption_A_image_only,caption_B_conditioned,error_A,error_B
22,51C1pOjJFaL__US,B074H6ZXG3,89ef5921,US,"/Categories/Meat & Seafood/Hot Dogs & Franks/Beef whole foods, whole food, Whole Foods,365 Everyday Value 365 by Whole Foods Market 365 Everyday Value, Uncured Beef Hot Dogs 6:1, 16 oz GROCERY Brought to you by Whole Foods Market. The packaging for this product has a fresh new look. During this transition, you may get the original packaging or the new packaging in your order, but the product and quality is staying exactly the same. Enjoy!",the product is shown in the image,"categories meat & seafood hot dogs & franks beef whole foods, whole food, whole foods, 365 everyday value 365 by whole foods market 365 everyday value, uncured beef hot dogs 6 : 1, 16 oz grocery brought to you by whole foods market. the packaging for this product has a fresh new look. during this transition, you may get the original packaging or the new packaging in your order, but the product and quality is staying exactly the same. enjoy!",,
29,81+iceLNMSL__US,B072V7HFS5,55d479cd,US,/Departments/Women/Shops christmas new years eve party Nia Knee-high Ankle Tie Boot The Fix Amazon Brand - The Fix Women's Nia Knee-High Ankle Tie Boot SHOES This knee-high boot distinguishes itself with an oval-shape heel and knot at the ankle.,a pair of brown boots with a bow on the side,departments women shops christmas new years eve party nia knee - high ankle tie boot the fix amazon brand - the fix women ' s nia knee - high ankle tie boot shoes this knee - high boot distinguishes itself with an oval - shape heel and knot at the ankle.,,
37,713TP3xS-TL__US,B07WZZL1J6,3db6494a,US,"PF-CS348-BR /Categories/Patio Furniture & Accessories/Patio Furniture Sets/Conversation Sets studio AmazonBasics AmazonBasics PF-CS348-BR Conversation Set, 4-Piece, Brown OUTDOOR_LIVING Brown 4-piece patio set includes: 1 love seat, 2 chairs, and 1 glass-top coffee table","the outdoor furniture set includes a sofa, chair and coffee table","pf - cs348 - br categories patio furniture & accessories patio furniture sets conversation sets studio amazonbasics amazonbasics pf - cs348 - br conversation set, 4 - piece, brown outdoor _ living brown 4 - piece patio set includes : 1 love seat, 2 chairs, and 1 glass - top coffee table",,
42,417WT9Pdf4L__US,B084JKB92S,7159d838,US,Fresh Pull Apart Onion Bread GROCERY,a white bowl with a small bowl of bread,"fresh pull apart onion bread grocery, the fresh pull apart, fresh pull apart, fresh pull apart, fresh pull apart, fresh pull apart, fresh pull apart, fresh pull apart",,
45,81GZIwFBJgL__US,B0841RDYCC,c3864541,US,"/Categories/Breads & Bakery/Pastries & Bakery/Muffins muffins Fresh Fresh Brand – Chocolate Chip Muffins, 14 oz (4 ct) CAKE 14-ounce package of 4 chocolate chip muffins",the fresh market chocolate chip muffins,"categories breads & bakery pastries & bakery muffins muffins fresh fresh brand – chocolate chip muffins, 14 oz ( 4 ct ) cake 14 - ounce package of 4 chocolate chip muffins",,
51,71uh2feF1VL__US,B01FGK95DQ,04976f99,US,"/Products/Hair Care/Shampoo & Conditioner/Shampoos Body Care Beauty Hair Care ,Beauty & Personal Care Hair Care Shampoo & Conditioner Shampoos 365 by Whole Foods Market 365 Everyday Value, Lavender Shampoo, 32 fl oz SHAMPOO Brought to you by Whole Foods Market. The packaging for this product has a fresh new look. During this transition, you may get the original packaging or the new packaging in your order, but the product and quality is staying exactly the same. Enjoy!",the body shop sham sham sham sham sham sham sham sham sham sham sham sham sham sham sham sham sham sham,"products hair care shampoo & conditioner shampoos body care beauty hair care, beauty & personal care hair care shampoo & conditioner shampoos 365 by whole foods market 365 everyday value, lavender shampoo, 32 fl oz shampoo brought to you by whole foods market. the packaging for this product has a fresh ne

In [ ]:
# Create a static instruction column for the df to be able to guide the BLIP text
master_metadata_df['static_instructions'] = 'Write a short, natural product caption for a shopper. Mention color, type, and key attributes. Keep it under 12 words.\n\n'

# Check DF
master_metadata_df.head()

,item_id,main_image_id,country,marketplace,domain_name,other_image_id,brand_extracted,bullet_point_extracted,color_extracted,style_extracted,material_extracted,item_keywords_extracted,node_extracted,item_name_extracted,item_weight_extracted,model_number_extracted,product_type_extracted,model_name_extracted,locale_identifier,blip_text_input,embedding_id,image_id,height,width,path,image_name,image_path,blip_text_input_clean,embedding_model_version,is_hero,static_instructions,instructions_plus_blip_text,instructions_plus_clean_blip_text,instructions_plus_clean_blip_input_text,blip_text_light,blip_text_light_clean
0,B07BMPCX9R,A1k2-zO8c+L,JP,Amazon,amazon.co.jp,"['51zE9cCrYiL', '812clWV8CPL', '51qVOaifn5L', '81ASk0Y1p7L', '51zqYmedAlL', '81YvgZoFHwL', '91XMjPo6kkL', '513Rtjm3siL', '81IfbHUqYbL', '91DmBO-9bSL', '71XSD2VTBmL']",Wickedly Prime (ウィキッドリープライム),"Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",NaN,NaN,NaN,パントリー対象商品,/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰,"[Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes)",50.00,NaN,GROCERY,NaN,JP,"/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",A1k2-zO8c+L__JP,A1k2-zO8c+L,2534,2560,72/72e64ca0.jpg,72e64ca0,/home/jovyan/data/shop_talk_data/image_data/images/small/72/72e6

In [ ]:
# Combine the BLIP instructions with the cleaned BLIP text
master_metadata_df['instructions_plus_clean_blip_input_text'] = master_metadata_df['static_instructions'] + master_metadata_df['blip_text_input_clean']

# Check DF
master_metadata_df.head()

,item_id,main_image_id,country,marketplace,domain_name,other_image_id,brand_extracted,bullet_point_extracted,color_extracted,style_extracted,material_extracted,item_keywords_extracted,node_extracted,item_name_extracted,item_weight_extracted,model_number_extracted,product_type_extracted,model_name_extracted,locale_identifier,blip_text_input,embedding_id,image_id,height,width,path,image_name,image_path,blip_text_input_clean,embedding_model_version,is_hero,static_instructions,instructions_plus_blip_text,instructions_plus_clean_blip_text,instructions_plus_clean_blip_input_text,blip_text_light,blip_text_light_clean
0,B07BMPCX9R,A1k2-zO8c+L,JP,Amazon,amazon.co.jp,"['51zE9cCrYiL', '812clWV8CPL', '51qVOaifn5L', '81ASk0Y1p7L', '51zqYmedAlL', '81YvgZoFHwL', '91XMjPo6kkL', '513Rtjm3siL', '81IfbHUqYbL', '91DmBO-9bSL', '71XSD2VTBmL']",Wickedly Prime (ウィキッドリープライム),"Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",NaN,NaN,NaN,パントリー対象商品,/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰,"[Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes)",50.00,NaN,GROCERY,NaN,JP,"/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",A1k2-zO8c+L__JP,A1k2-zO8c+L,2534,2560,72/72e64ca0.jpg,72e64ca0,/home/jovyan/data/shop_talk_data/image_data/images/small/72/72e6

## Test Image vs Conditioned Text vs Instructructed Text

In [ ]:
# ====== 3A) A/B/C caption test (image-only vs conditioned vs instruction seasoned conditioned) ======

import pandas as pd
import numpy as np
import os

AB_SAMPLE_N = 100
AB_RANDOM_SEED = 42
ABC_OUT_CSV = os.path.join(VERSION_DIR, "abc_caption_test.csv")

# Use the full master metadata (includes country, locale, etc.)
ab_df = master_metadata_df.copy()
ab_df["embedding_id"] = ab_df["embedding_id"].astype(str)

# Sample rows that actually have image paths present
ab_df = ab_df[ab_df["image_path"].notna()].copy()

sample_df = ab_df.sample(n=min(AB_SAMPLE_N, len(ab_df)), random_state=AB_RANDOM_SEED).reset_index(drop=True)

rows = []
for row in sample_df.itertuples(index=False):
    try:
        cap_img = generate_image_caption(row.image_path)
        err_img = ""
    except Exception as e:
        cap_img = ""
        err_img = str(e)

    try:
        cap_cond = generate_conditional_caption(row.image_path, str(row.blip_text_input_clean))
        err_cond = ""
    except Exception as e:
        cap_cond = ""
        err_cond = str(e)


    try:
        instrc_cap_cond = generate_conditional_caption(row.image_path, str(row.instructions_plus_clean_blip_input_text))
        instrc_err_cond = ""
    except Exception as e:
        instrc_cap_cond = ""
        instrc_err_cond = str(e)


    rows.append({
        "embedding_id": str(row.embedding_id),
        "item_id": getattr(row, "item_id", None),
        "image_name": getattr(row, "image_name", None),
        "country": getattr(row, "country", None),
        "blip_text_input": str(getattr(row, "blip_text_input", "")),
        "caption_A_image_only": cap_img,
        "caption_B_conditioned": cap_cond,
        "caption_C_conditioned": instrc_cap_cond,
        "error_A": err_img,
        "error_B": err_cond,
        "error_C": instrc_err_cond,
    })

abc_out = pd.DataFrame(rows)
abc_out.to_csv(ABC_OUT_CSV, index=False)
print("Wrote A/B/C caption test:", ABC_OUT_CSV)
abc_out.head(10)

Wrote A/B/C caption test: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/abc_caption_test.csv


,embedding_id,item_id,image_name,country,blip_text_input,caption_A_image_only,caption_B_conditioned,caption_C_conditioned,error_A,error_B,error_C
0,71H15JG9t-L__IN,B08545ZY6B,5113fcba,IN,"Vivo Z1 Pro /Categories/Mobiles & Accessories/Mobile Accessories/Maintenance, Upkeep & Repairs/Replacement Parts/Back Covers Back Cover Amazon Brand - Solimo Amazon Brand - Solimo Designer Photography UV Printed Soft Back Case Mobile Cover for Vivo Z1 Pro CELLULAR_PHONE_CASE Multicolor Silicon Snug fit for Vivo Z1 Pro, with perfect cut-outs for volume buttons, audio and charging ports",a phone case with a unicorn and a unicorn on it,"vivo z1 pro categories mobiles & accessories mobile accessories maintenance, upkeep & repairs replacement parts back covers back cover amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro cellular _ phone _ case multicolor silicon snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports","write a short, natural product caption for a shopper. mention color, type, and key attributes. keep it under 12 words. vivo z1 pro categories mobiles & accessories mobile accessories maintenance, upkeep & repairs replacement parts back covers back cover amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro cellular _ phone _ case multicolor silicon snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports",,,
1,915+S1MRS4L__SG,B07PHFWND8,fa72077d,SG,"Sheet Set - Twin, Sage Vine /Homeware & Furniture/Bedding/Sheets & Pillowcases/Sheet & Pillowcase Sets bed AmazonBasics AmazonBasics Super-Soft Cotton Bed Sheet Set-Twin, Sage Vine FLAT_SHEET Sage Vine 100% Cotton Twin set contains one 68 x 96 inch flat sheet; one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",the white and green floral bedding set,"sheet set - twin, sage vine homeware & furniture bedding sheets & pillowcases sheet & pillowcase sets bed amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine flat _ sheet sage vine 100 % cotton twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases","write a short, natural product caption for a shopper. mention color, type, and key attributes. keep it under 12 words. sheet set - twin, sage vine homeware & furniture bedding sheets & pillowcases sheet & pillowcase sets bed amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine flat _ sheet sage vine 100 % cotton twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",,,
2,617Gg5OD-TL__IN,B0856G38P5,54462c3c,IN,"Lenovo P2 /Categories/Mobiles & Accessories/Mobile Accessories/Cases & Covers/Back & Bumper Cases cellphonecover Amazon Brand - Solimo Amazon Brand - Solimo Designer Heart Pattern Alphabet-H 3D Printed Hard Back Case Mobile Cover for Lenovo P2 CELLULAR_PHONE_CASE multi-colored Plastic Snug fit for Mobile, with perfect cut-outs for volume buttons, audio and charging ports",the h logo on a pink and black phone case,"lenovo p2 categories mobiles & accessories mobile accessories cases & covers back & bumper cases cellphonecover amazon brand - solimo amazon brand - solimo designer heart pattern alphabet - h 3d printed hard back case mobile cover for lenovo p2 cellular _ phone _ case multi - colored plastic snug fit for mobile, with perfect cut - outs for volume buttons, audio and charging ports","write a short, natural product caption for a shopper. mention color, type, and key attributes. keep it under 12 words. lenovo p2 categories mobiles & accessories mobile accessories cases & covers back & bumper cases cellphonecover amazon brand - solimo amazon brand - solimo designer heart pattern alphabet - h 3d printed hard back case mobile cover for lenovo p2 cellular _ phone _ case multi - colored plas

## Test Image vs Conditioned Text vs Instructructed Text vs Light Blip Text

In [ ]:
# Define the columns to concatenate
semantic_fields = [
    'style_extracted',
    'brand_extracted',
    'item_name_extracted',
    'color_extracted',
    'bullet_point_extracted'
]

#Create a new blip_text_light colun in the master_metadata_df that is a combination of the values in the semantic_fields columns for each row
master_metadata_df['blip_text_light'] = master_metadata_df[semantic_fields].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)

# Check DF
master_metadata_df.head()

,item_id,main_image_id,country,marketplace,domain_name,other_image_id,brand_extracted,bullet_point_extracted,color_extracted,style_extracted,material_extracted,item_keywords_extracted,node_extracted,item_name_extracted,item_weight_extracted,model_number_extracted,product_type_extracted,model_name_extracted,locale_identifier,blip_text_input,embedding_id,image_id,height,width,path,image_name,image_path,blip_text_input_clean,embedding_model_version,is_hero,static_instructions,instructions_plus_blip_text,instructions_plus_clean_blip_text,instructions_plus_clean_blip_input_text,blip_text_light,blip_text_light_clean
0,B07BMPCX9R,A1k2-zO8c+L,JP,Amazon,amazon.co.jp,"['51zE9cCrYiL', '812clWV8CPL', '51qVOaifn5L', '81ASk0Y1p7L', '51zqYmedAlL', '81YvgZoFHwL', '91XMjPo6kkL', '513Rtjm3siL', '81IfbHUqYbL', '91DmBO-9bSL', '71XSD2VTBmL']",Wickedly Prime (ウィキッドリープライム),"Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",NaN,NaN,NaN,パントリー対象商品,/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰,"[Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes)",50.00,NaN,GROCERY,NaN,JP,"/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",A1k2-zO8c+L__JP,A1k2-zO8c+L,2534,2560,72/72e64ca0.jpg,72e64ca0,/home/jovyan/data/shop_talk_data/image_data/images/small/72/72e6

In [ ]:
# Create additive cleaned column (do not overwrite raw)
master_metadata_df["blip_text_light_clean"] = master_metadata_df["blip_text_light"].apply(clean_blip_text)

# Check df
master_metadata_df.head()

,item_id,main_image_id,country,marketplace,domain_name,other_image_id,brand_extracted,bullet_point_extracted,color_extracted,style_extracted,material_extracted,item_keywords_extracted,node_extracted,item_name_extracted,item_weight_extracted,model_number_extracted,product_type_extracted,model_name_extracted,locale_identifier,blip_text_input,embedding_id,image_id,height,width,path,image_name,image_path,blip_text_input_clean,embedding_model_version,is_hero,static_instructions,instructions_plus_blip_text,instructions_plus_clean_blip_text,instructions_plus_clean_blip_input_text,blip_text_light,blip_text_light_clean
0,B07BMPCX9R,A1k2-zO8c+L,JP,Amazon,amazon.co.jp,"['51zE9cCrYiL', '812clWV8CPL', '51qVOaifn5L', '81ASk0Y1p7L', '51zqYmedAlL', '81YvgZoFHwL', '91XMjPo6kkL', '513Rtjm3siL', '81IfbHUqYbL', '91DmBO-9bSL', '71XSD2VTBmL']",Wickedly Prime (ウィキッドリープライム),"Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",NaN,NaN,NaN,パントリー対象商品,/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰,"[Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes)",50.00,NaN,GROCERY,NaN,JP,"/カテゴリー別/缶詰・瓶詰/肉の缶詰・瓶詰 パントリー対象商品 Wickedly Prime (ウィキッドリープライム) [Amazon Brand] Wickedly Prime Premium Pinch Set of 3 (Hormone Beef Marchows, Hormone Chicken Straight Fire, Hormone Chicken Serpes) GROCERY Ingredients: [Wickedly Prime Prime Prime Snack Hormone Beef Marchel Grilled Grilled with Beef] Small intestine (including beef), Cochjjang (including soybean), Miso, Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soybeans, mackerage)), wheat flour, bean plate sauce, garlic, ginger, sugar, powdered food oil, salt, and broth Spices/thickener (processed starch, polysaccharide thickener), sorbitol, glycine, Na acetate, seasoning (amino acids, etc.) 【Wickedly Prime Premium Snack Hormone Chicken Flami】Chicken Spears (including Chicken), Salt (salt, spices, sugar, garlic), Spices, Spices, Spices, Spices, Oil and Sesame Oil, Whey Powder (including milk ingredients) / Seasoning (amino acids), Na phosphate, thickener (processed starch, polysaccharide thickener), glycine, acetate Na, pH regulator, lactic acid Ca, emulsifier, antioxidant (V.C, V.E), flavoring, carotenoid pigment [Wickedly Premium Snack hormone, chicken seri, direct fire] Chicken serries (including chicken), Sauce (soy sauce, sugar, fermented seasonings, seafood extract (including wheat, soy, mackerel), spices, ginger, salt, whey powder (including milk ingredients) / sorbitol, seasonings (organic acids), thickening agents (processed starch, polysaccharids), caramel pigment, glycin, Na acetate",A1k2-zO8c+L__JP,A1k2-zO8c+L,2534,2560,72/72e64ca0.jpg,72e64ca0,/home/jovyan/data/shop_talk_data/image_data/images/small/72/72e6

In [ ]:
# ====== 3A) A/B/C caption test (image-only vs conditioned vs instruction seasoned conditioned) ======

import pandas as pd
import numpy as np
import os

AB_SAMPLE_N = 100
AB_RANDOM_SEED = 42
ABCD_OUT_CSV = os.path.join(VERSION_DIR, "abcd_caption_test.csv")

# Use the full master metadata (includes country, locale, etc.)
ab_df = master_metadata_df.copy()
ab_df["embedding_id"] = ab_df["embedding_id"].astype(str)

# Sample rows that actually have image paths present
ab_df = ab_df[ab_df["image_path"].notna()].copy()

sample_df = ab_df.sample(n=min(AB_SAMPLE_N, len(ab_df)), random_state=AB_RANDOM_SEED).reset_index(drop=True)

rows = []
for row in sample_df.itertuples(index=False):
    try:
        cap_img = generate_image_caption(row.image_path)
        err_img = ""
    except Exception as e:
        cap_img = ""
        err_img = str(e)


    try:
        cap_cond = generate_conditional_caption(row.image_path, str(row.blip_text_input_clean))
        err_cond = ""
    except Exception as e:
        cap_cond = ""
        err_cond = str(e)


    try:
        instrc_cap_cond = generate_conditional_caption(row.image_path, str(row.instructions_plus_clean_blip_input_text))
        instrc_err_cond = ""
    except Exception as e:
        instrc_cap_cond = ""
        instrc_err_cond = str(e)


    try:
        light_cap_cond = generate_conditional_caption(row.image_path, str(row.blip_text_light_clean))
        light_err_cond = ""
    except Exception as e:
        light_cap_cond = ""
        light_err_cond = str(e)


    rows.append({
        "embedding_id": str(row.embedding_id),
        "item_id": getattr(row, "item_id", None),
        "image_name": getattr(row, "image_name", None),
        "country": getattr(row, "country", None),
        "blip_text_input": str(getattr(row, "blip_text_input", "")),
        "caption_A_image_only": cap_img,
        "caption_B_conditioned": cap_cond,
        "caption_C_conditioned": instrc_cap_cond,
        "caption_D_conditioned": light_cap_cond,
        "error_A": err_img,
        "error_B": err_cond,
        "error_C": instrc_err_cond,
        "error_D": light_err_cond,
    })

abcd_out = pd.DataFrame(rows)
abcd_out.to_csv(ABCD_OUT_CSV, index=False)
print("Wrote A/B/C caption test:", ABCD_OUT_CSV)
abcd_out.head(20)

Wrote A/B/C caption test: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/abcd_caption_test.csv


,embedding_id,item_id,image_name,country,blip_text_input,caption_A_image_only,caption_B_conditioned,caption_C_conditioned,caption_D_conditioned,error_A,error_B,error_C,error_D
0,71H15JG9t-L__IN,B08545ZY6B,5113fcba,IN,"Vivo Z1 Pro /Categories/Mobiles & Accessories/Mobile Accessories/Maintenance, Upkeep & Repairs/Replacement Parts/Back Covers Back Cover Amazon Brand - Solimo Amazon Brand - Solimo Designer Photography UV Printed Soft Back Case Mobile Cover for Vivo Z1 Pro CELLULAR_PHONE_CASE Multicolor Silicon Snug fit for Vivo Z1 Pro, with perfect cut-outs for volume buttons, audio and charging ports",a phone case with a unicorn and a unicorn on it,"vivo z1 pro categories mobiles & accessories mobile accessories maintenance, upkeep & repairs replacement parts back covers back cover amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro cellular _ phone _ case multicolor silicon snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports","write a short, natural product caption for a shopper. mention color, type, and key attributes. keep it under 12 words. vivo z1 pro categories mobiles & accessories mobile accessories maintenance, upkeep & repairs replacement parts back covers back cover amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro cellular _ phone _ case multicolor silicon snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports","amazon brand - solimo amazon brand - solimo designer photography uv printed soft back case mobile cover for vivo z1 pro multicolor snug fit for vivo z1 pro, with perfect cut - outs for volume buttons, audio and charging ports",,,,
1,915+S1MRS4L__SG,B07PHFWND8,fa72077d,SG,"Sheet Set - Twin, Sage Vine /Homeware & Furniture/Bedding/Sheets & Pillowcases/Sheet & Pillowcase Sets bed AmazonBasics AmazonBasics Super-Soft Cotton Bed Sheet Set-Twin, Sage Vine FLAT_SHEET Sage Vine 100% Cotton Twin set contains one 68 x 96 inch flat sheet; one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",the white and green floral bedding set,"sheet set - twin, sage vine homeware & furniture bedding sheets & pillowcases sheet & pillowcase sets bed amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine flat _ sheet sage vine 100 % cotton twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases","write a short, natural product caption for a shopper. mention color, type, and key attributes. keep it under 12 words. sheet set - twin, sage vine homeware & furniture bedding sheets & pillowcases sheet & pillowcase sets bed amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine flat _ sheet sage vine 100 % cotton twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases","amazonbasics amazonbasics super - soft cotton bed sheet set - twin, sage vine sage vine twin set contains one 68 x 96 inch flat sheet, one 39 x 75 x 16 inch fitted sheet, and two 21 x 32 inch pillowcases",,,,
2,617Gg5OD-TL__IN,B0856G38P5,54462c3c,IN,"Lenovo P2 /Categories/Mobiles & Accessories/Mobile Accessories/Cases & Covers/Back & Bumper Cases cellphonecover Amazon Brand - Solimo Amazon Brand - Solimo Designer Heart Pattern Alphabet-H 3D Printed Hard Back Case Mobile Cover for Lenovo P2 CELLULAR_PHONE_CASE multi-colored Plastic Snug fit for Mobile, with perfect cut-outs for volume buttons, audio and charging ports",the h logo on a pink and black phone case,"lenovo p2 categories mobiles & accessories mobile accessories cases & covers back & bumper cases cellphonecover amazon brand - solimo amazon brand - solimo designer heart pattern alphabet - h 3d printed hard back case mobile cover for lenovo p2 cellular _ phone _ case multi - colored plastic snug fit for mobile, with perfect 

In [ ]:
# ====== 4) Generate BLIP captions (resumable; additive metadata) ======
# Commentary:
# - BLIP here is used in *conditional captioning* mode.
# - We feed BLIP two inputs:
#   (1) the product image
#   (2) `blip_text_input` (metadata prompt) to steer caption generation toward relevant attributes.
# - The output is a short caption string we later embed in CLIP text space for retrieval.
#
# Key improvements in this version:
# 1) **Additive output**: every output row includes ALL original columns from `master_metadata_df`
#    (e.g., country/locale/price/etc.) plus new columns: `blip_caption`, `caption_error`.
# 2) **Correct resume logic**: empty-string captions '' are NOT treated as done.
#    Done means: blip_caption is non-empty after stripping whitespace.
# 3) Optional retries: by default we retry blank captions (including prior failures).

import os
import pandas as pd

RETRY_PREVIOUS_FAILURES = True   # if True, reprocess rows with blank captions
flush_every = 200

# Ensure consistent types
work_df = master_metadata_df.copy()
work_df["embedding_id"] = work_df["embedding_id"].astype(str)

# Columns we will always add (even if not present in the master table)
NEW_COLS = ["blip_caption", "caption_error"]

# -----------------------------
# 1) Load prior enriched table (if present) and compute DONE set correctly
# -----------------------------
if os.path.exists(MASTER_ENRICHED_OUT_CSV):
    enriched = pd.read_csv(MASTER_ENRICHED_OUT_CSV)
    enriched["embedding_id"] = enriched["embedding_id"].astype(str)

    # "Good caption" = non-empty after stripping
    cap_good = enriched.get("blip_caption", pd.Series([""] * len(enriched))).fillna("").astype(str).str.strip().ne("")

    # Done definition
    if RETRY_PREVIOUS_FAILURES:
        done_mask = cap_good
    else:
        err_bad = enriched.get("caption_error", pd.Series([""] * len(enriched))).fillna("").astype(str).str.strip().ne("")
        done_mask = cap_good & (~err_bad)

    done = set(enriched.loc[done_mask, "embedding_id"].astype(str))

    # Diagnostics
    print("Found existing master_metadata_enriched rows:", len(enriched))
    print("Good captions (done):", int(cap_good.sum()))
    print("Blank captions (will retry):", int((~cap_good).sum()))
else:
    enriched = None
    done = set()
    print("No existing enriched CSV found. Starting fresh.")

# -----------------------------
# 2) Build TODO set
# -----------------------------
todo_df = work_df[~work_df["embedding_id"].isin(done)].copy()
print("Captions to generate:", len(todo_df))

# If nothing left, still snapshot to Parquet
if len(todo_df) == 0:
    export_parquet_from_csv(MASTER_ENRICHED_OUT_CSV, MASTER_ENRICHED_OUT_PQ)

else:
    # If we already have an enriched file, lock the column order to avoid append misalignment
    existing_cols = None
    if enriched is not None:
        existing_cols = list(enriched.columns)

    # Ensure the master table has the new columns (as placeholders) so output schema is stable
    for col in NEW_COLS:
        if col not in work_df.columns:
            work_df[col] = ""

    # -----------------------------
    # 3) Caption loop with periodic flushing
    # -----------------------------
    rows_out = []

    for i, row in enumerate(todo_df.itertuples(index=False), start=1):
        # Build a base row that includes ALL original metadata columns
        base = row._asdict()
        base["embedding_id"] = str(base.get("embedding_id", ""))

        try:
            cap = generate_conditional_caption(base["image_path"], str(base.get("blip_text_input", "")))
            cap = (cap or "").strip()
            err = ""
        except Exception as e:
            cap = ""
            err = str(e)

        base["blip_caption"] = cap
        base["caption_error"] = err

        rows_out.append(base)

        # Flush
        if (i % flush_every) == 0:
            out_df = pd.DataFrame(rows_out)

            # If appending, enforce identical column order
            if existing_cols is not None:
                out_df = out_df.reindex(columns=existing_cols)
            else:
                # first write: ensure new cols are present and move them to the end for readability
                cols = [c for c in out_df.columns if c not in NEW_COLS] + NEW_COLS
                out_df = out_df.reindex(columns=cols)
                existing_cols = list(out_df.columns)

            if os.path.exists(MASTER_ENRICHED_OUT_CSV):
                out_df.to_csv(MASTER_ENRICHED_OUT_CSV, mode="a", header=False, index=False)
            else:
                out_df.to_csv(MASTER_ENRICHED_OUT_CSV, index=False)

            print(f"Flushed {len(out_df)} rows at checkpoint i={i}.")
            rows_out = []

    # Final flush
    if rows_out:
        out_df = pd.DataFrame(rows_out)
        if existing_cols is not None:
            out_df = out_df.reindex(columns=existing_cols)
        else:
            cols = [c for c in out_df.columns if c not in NEW_COLS] + NEW_COLS
            out_df = out_df.reindex(columns=cols)
            existing_cols = list(out_df.columns)

        if os.path.exists(MASTER_ENRICHED_OUT_CSV):
            out_df.to_csv(MASTER_ENRICHED_OUT_CSV, mode="a", header=False, index=False)
        else:
            out_df.to_csv(MASTER_ENRICHED_OUT_CSV, index=False)

        print(f"Final flush wrote {len(out_df)} rows.")

    print("Done. Wrote:", MASTER_ENRICHED_OUT_CSV)

    # -----------------------------
    # 4) Canonical Parquet snapshot
    # -----------------------------
    export_parquet_from_csv(MASTER_ENRICHED_OUT_CSV, MASTER_ENRICHED_OUT_PQ)

Found existing master_metadata_enriched rows: 200233
Captions to generate: 0
Done. Wrote: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/master_metadata_enriched.csv
Wrote Parquet: /home/jovyan/data/shop_talk_data/model_outputs_v3_itemcentric/v1_clip-vit-b32__blip-base__caption2clip/master_metadata_enriched.parquet | rows: 200233


In [ ]:
# ====== 5) CLIP setup (text + image embeddings) ======
# Commentary:
# - We normalize embeddings to unit length so cosine similarity becomes a simple dot product.
# - This is important because FAISS will use IndexFlatIP (inner product) to approximate cosine similarity.

import os, numpy as np, pandas as pd, torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_model.eval()

def clip_text_embed(text: str) -> np.ndarray:
    inputs = clip_processor(text=[text], return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        feats = clip_model.get_text_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats[0].detach().cpu().numpy().astype('float32')

def clip_image_embed(image_path: str) -> np.ndarray:
    image = Image.open(image_path).convert('RGB')
    inputs = clip_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        feats = clip_model.get_image_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats[0].detach().cpu().numpy().astype('float32')

def caption_emb_path(embedding_id: str) -> str:
    return os.path.join(CLIP_CAPTION_EMB_DIR, f'{embedding_id}.npy')

def image_emb_path(embedding_id: str) -> str:
    return os.path.join(CLIP_IMAGE_EMB_DIR, f'{embedding_id}.npy')

In [ ]:
# ====== 6) Generate CLIP caption embeddings from BLIP captions (resumable) ======
# Commentary:
# - We normalize embeddings to unit length so cosine similarity becomes a simple dot product.
# - This is important because FAISS will use IndexFlatIP (inner product) to approximate cosine similarity.

import os, pandas as pd, numpy as np

enriched = pd.read_csv(MASTER_ENRICHED_OUT_CSV)
enriched['embedding_id'] = enriched['embedding_id'].astype(str)

# Only embed captions that exist and haven't been embedded yet
def has_caption_vec(eid: str) -> bool:
    return os.path.exists(caption_emb_path(eid))

todo = enriched[(enriched['blip_caption'].fillna('') != '') & (~enriched['embedding_id'].apply(has_caption_vec))].copy()
print('Caption embeddings to generate:', len(todo))

flush_every = 500
count = 0

for row in todo.itertuples(index=False):
    try:
        vec = clip_text_embed(str(row.blip_caption))
        np.save(caption_emb_path(row.embedding_id), vec)
    except Exception as e:
        with open(os.path.join(CLIP_CAPTION_EMB_DIR, f'{row.embedding_id}.error.txt'), 'w') as f:
            f.write(str(e))
    count += 1
    if count % flush_every == 0:
        print('Generated caption embeddings:', count)

print('Done. Caption embedding dir:', CLIP_CAPTION_EMB_DIR)

# Update master enriched table with paths (cheap + fast)
enriched['clip_caption_embedding_path'] = enriched['embedding_id'].apply(lambda x: caption_emb_path(str(x)) if os.path.exists(caption_emb_path(str(x))) else '')
enriched.to_csv(MASTER_ENRICHED_OUT_CSV, index=False)
print('Updated master enriched with clip_caption_embedding_path')


# Parquet snapshot (canonical)
export_parquet_from_csv(MASTER_ENRICHED_OUT_CSV, MASTER_ENRICHED_OUT_PQ)

In [ ]:
# ====== 7) Generate CLIP image embeddings (kept in a separate table) ======
# Commentary:
# - CSV is our interruption-safe checkpoint format (append-friendly).
# - Parquet is our canonical analytics/storage format (typed, compressed, fast).
# - This helper converts the latest CSV checkpoint into a Parquet snapshot whenever we finish a stage.

import os, pandas as pd, numpy as np

enriched = pd.read_csv(MASTER_ENRICHED_OUT_CSV)
enriched['embedding_id'] = enriched['embedding_id'].astype(str)

def has_image_vec(eid: str) -> bool:
    return os.path.exists(image_emb_path(eid))

todo = enriched[~enriched['embedding_id'].apply(has_image_vec)].copy()
print('Image embeddings to generate:', len(todo))

flush_every = 500
count = 0

for row in todo.itertuples(index=False):
    try:
        vec = clip_image_embed(row.image_path)
        np.save(image_emb_path(row.embedding_id), vec)
    except Exception as e:
        with open(os.path.join(CLIP_IMAGE_EMB_DIR, f'{row.embedding_id}.error.txt'), 'w') as f:
            f.write(str(e))
    count += 1
    if count % flush_every == 0:
        print('Generated image embeddings:', count)

print('Done. Image embedding dir:', CLIP_IMAGE_EMB_DIR)

# Create the separate image-embedding table you asked for
img_table = enriched[['embedding_id','item_id','image_id','image_name','image_path','is_hero']].copy()
img_table['clip_image_embedding_path'] = img_table['embedding_id'].apply(lambda x: image_emb_path(str(x)) if os.path.exists(image_emb_path(str(x))) else '')
img_table.to_csv(CLIP_IMAGE_TABLE_OUT_CSV, index=False)
print('Wrote CLIP image embedding table:', CLIP_IMAGE_TABLE_OUT_CSV)
print(img_table.head(3))


# Parquet snapshot (canonical)
export_parquet_from_csv(CLIP_IMAGE_TABLE_OUT_CSV, CLIP_IMAGE_TABLE_OUT_PQ)

In [ ]:
# ====== 8) Item-level aggregation (caption-embedding centroid; hero-weighted) ======
# Commentary:
# - CSV is our interruption-safe checkpoint format (append-friendly).
# - Parquet is our canonical analytics/storage format (typed, compressed, fast).
# - This helper converts the latest CSV checkpoint into a Parquet snapshot whenever we finish a stage.

import os, numpy as np, pandas as pd

enriched = pd.read_csv(MASTER_ENRICHED_OUT_CSV)
enriched['embedding_id'] = enriched['embedding_id'].astype(str)

def load_caption_vec(eid: str) -> np.ndarray:
    return np.load(caption_emb_path(str(eid)))

rows = []
for item_id, g in enriched.groupby('item_id'):
    # load vectors that exist
    vecs = []
    hero_vec = None

    for r in g.itertuples(index=False):
        p = caption_emb_path(str(r.embedding_id))
        if os.path.exists(p):
            v = np.load(p)
            vecs.append(v)
            if bool(r.is_hero):
                hero_vec = v

    if not vecs:
        continue

    vecs = np.stack(vecs, axis=0)
    mean_vec = vecs.mean(axis=0)

    if hero_vec is not None:
        item_vec = 0.6*hero_vec + 0.4*mean_vec
    else:
        item_vec = mean_vec

    item_vec = item_vec / (np.linalg.norm(item_vec) + 1e-12)

    item_vec_path = os.path.join(ITEM_EMB_DIR, f'{item_id}.npy')
    np.save(item_vec_path, item_vec.astype('float32'))

    rows.append({
        'item_id': item_id,
        'n_images': len(g),
        'main_image_id': g['image_id'][g['is_hero']].iloc[0] if (g['is_hero'].sum() > 0) else g['image_id'].iloc[0],
        'embedding_ids': list(g['embedding_id'].astype(str).unique()),
        'item_embedding_path': item_vec_path
    })

item_index = pd.DataFrame(rows)
item_index.to_csv(ITEM_LEVEL_OUT_CSV, index=False)
print('Wrote item index:', ITEM_LEVEL_OUT_CSV)
print(item_index.head(3))


# Parquet snapshot (canonical)
export_parquet_from_csv(ITEM_LEVEL_OUT_CSV, ITEM_LEVEL_OUT_PQ)

In [ ]:
# ====== 9) Retrieval demo (query -> top item_ids using item caption-embeddings) ======
# Commentary:
# - We normalize embeddings to unit length so cosine similarity becomes a simple dot product.
# - This is important because FAISS will use IndexFlatIP (inner product) to approximate cosine similarity.

import numpy as np, pandas as pd, os

item_index = pd.read_csv(ITEM_LEVEL_OUT_CSV)

# Load item vectors in-memory for a demo. For production: FAISS.
item_vecs, item_ids = [], []
for r in item_index.itertuples(index=False):
    if os.path.exists(r.item_embedding_path):
        item_vecs.append(np.load(r.item_embedding_path))
        item_ids.append(r.item_id)

item_vecs = np.stack(item_vecs, axis=0)

def search_items(query: str, top_k: int = 10):
    q = clip_text_embed(query)
    sims = item_vecs @ q
    top = np.argsort(-sims)[:top_k]
    return list(zip([item_ids[i] for i in top], sims[top]))

print(search_items('get me red shirts under $50', top_k=10))

In [ ]:
# ====== 10) Build FAISS item-level index (fast retrieval) ======
# Notes:
# - We use cosine similarity by indexing *normalized* vectors in an Inner Product (IP) index.
# - We also write an id-map so FAISS positions map back to item_id.

import os, numpy as np, pandas as pd

# Install FAISS in Colab if missing
try:
    import faiss
except Exception:
    !pip -q install faiss-cpu
    import faiss

item_index = pd.read_csv(ITEM_LEVEL_OUT_CSV)

vecs = []
ids = []
for r in item_index.itertuples(index=False):
    p = r.item_embedding_path
    if os.path.exists(p):
        v = np.load(p).astype('float32')
        # Ensure normalized
        v = v / (np.linalg.norm(v) + 1e-12)
        vecs.append(v)
        ids.append(r.item_id)

assert len(vecs) > 0, "No item embeddings found to index."
X = np.stack(vecs, axis=0).astype('float32')

dim = X.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine via normalized vectors
index.add(X)

faiss.write_index(index, FAISS_ITEM_INDEX_PATH)

idmap = pd.DataFrame({'faiss_pos': np.arange(len(ids)), 'item_id': ids})
idmap.to_csv(FAISS_ITEM_IDMAP_PATH, index=False)

print('Wrote:', FAISS_ITEM_INDEX_PATH)
print('Wrote:', FAISS_ITEM_IDMAP_PATH)
print('Indexed item vectors:', X.shape)


In [ ]:
# ====== 11) Build FAISS image-level index (IMAGE-CENTRIC) ======
# Image-centric FAISS index:
# - Vectors: CLIP image embeddings (one per embedding_id)
# - Primary returned identifier: image_name
# - Still keeps embedding_id for traceability + file paths
#
# This supports:
# - Visual similarity search
# - Image reranking
# - Image-upload search
#
# FAISS positions map to *images*, not items.

import os, numpy as np, pandas as pd

try:
    import faiss
except Exception:
    !pip -q install faiss-cpu
    import faiss

img_table = pd.read_csv(CLIP_IMAGE_TABLE_OUT_CSV)
img_table['embedding_id'] = img_table['embedding_id'].astype(str)

vecs = []
rows = []

for r in img_table.itertuples(index=False):
    p = getattr(r, 'clip_image_embedding_path', '')
    if isinstance(p, str) and p and os.path.exists(p):
        v = np.load(p).astype('float32')
        v = v / (np.linalg.norm(v) + 1e-12)
        vecs.append(v)
        rows.append({
            'embedding_id': str(r.embedding_id),
            'image_name': r.image_name,
            'item_id': r.item_id,
            'image_id': r.image_id,
            'image_path': r.image_path,
            'is_hero': r.is_hero
        })

if not vecs:
    raise RuntimeError("No image embeddings found to index. Did you run the image embedding step?")

X = np.stack(vecs, axis=0).astype('float32')
dim = X.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(X)

faiss.write_index(index, FAISS_IMAGE_INDEX_PATH)

# IMAGE-CENTRIC ID MAP
idmap = pd.DataFrame(rows)
idmap.insert(0, 'faiss_pos', np.arange(len(idmap)))
idmap.to_csv(FAISS_IMAGE_IDMAP_PATH, index=False)

print('Wrote IMAGE-CENTRIC FAISS index:', FAISS_IMAGE_INDEX_PATH)
print('Wrote IMAGE-CENTRIC ID map:', FAISS_IMAGE_IDMAP_PATH)
print('Indexed image vectors:', X.shape)
print(idmap.head(3))


In [ ]:
# ====== 12) FAISS retrieval demo (item-level) ======
# Commentary:
# - We normalize embeddings to unit length so cosine similarity becomes a simple dot product.
# - This is important because FAISS will use IndexFlatIP (inner product) to approximate cosine similarity.

import numpy as np, pandas as pd

try:
    import faiss
except Exception:
    !pip -q install faiss-cpu
    import faiss

index = faiss.read_index(FAISS_ITEM_INDEX_PATH)
idmap = pd.read_csv(FAISS_ITEM_IDMAP_PATH)

def faiss_search_items(query: str, top_k: int = 10):
    q = clip_text_embed(query).astype('float32')
    q = q / (np.linalg.norm(q) + 1e-12)
    D, I = index.search(q.reshape(1,-1), top_k)
    out = []
    for score, pos in zip(D[0], I[0]):
        if pos < 0:
            continue
        item_id = idmap.loc[idmap['faiss_pos'] == pos, 'item_id'].values[0]
        out.append((item_id, float(score)))
    return out

print(faiss_search_items('get me red shirts under $50', top_k=10))